# 풀스택 GPT: #5.0~5.8

## Tasks:

- [ ] 앞서 배운 메모리 클래스 중 하나를 사용하는 메모리로 LCEL 체인을 구현합니다.
    > - [ ] [ConversationBufferMemory](https://python.langchain.com/v0.1/docs/modules/memory/types/buffer/) 등 강의에서 배운 메모리 중 하나를 사용하여 이전 대화 기록을 기억하고 기록을 이용한 답변을 제공할 수 있도록 합니다.
    > - [x] 채팅 형식의 메모리 기록을 프롬프트에 추가하고 싶을 때는 [MessagesPlaceholder](https://python.langchain.com/v0.1/docs/modules/memory/adding_memory/#adding-memory-to-a-chat-model-based-llmchain)를 이용하세요.
    > - [x] RunnablePassthrough를 활용하면 LCEL 체인을 구현할 때 메모리 적용을 쉽게 할 수 있습니다. RunnablePassthrough는 메모리를 포함한 데이터를 체인의 각 단계에 전달하는 역할을 합니다. (강의 #5.7 1:04~ 참고)
- [ ] 이 체인은 영화 제목을 가져와 영화를 나타내는 세 개의 이모티콘으로 응답해야 합니다. (예: "탑건" -> "🛩️👨‍✈️🔥". "대부" -> "👨‍👨‍👦🔫🍝").
- [ ] 항상 세 개의 이모티콘으로 답장하도록 [FewShotPromptTemplate](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples/) 또는 [FewShotChatMessagePromptTemplate](https://python.langchain.com/v0.1/docs/modules/model_io/prompts/few_shot_examples_chat/)을 사용하여 체인에 예시를 제공하세요.
- [x] 메모리가 작동하는지 확인하려면 체인에 두 개의 영화에 대해 질문한 다음 다른 셀에서 체인에 먼저 질문한 영화가 무엇인지 알려달라고 요청하세요.

In [14]:
examples = [
    {
        "question":"Top Gun",
        "answer":"🛩️👨‍✈️🔥",
    },
    {
        "question":"The Godfather",
        "answer":"👨‍👨‍👦🔫🍝"
    },
]

In [ ]:
import os
from dotenv import load_dotenv
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate, FewShotPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

load_dotenv()
set_llm_cache(InMemoryCache())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME"),
    temperature=1.0,
)
memory = ConversationBufferMemory()

few_shot_example_prompt = PromptTemplate.from_template("""
    Human: {question}
    AI: {answer}
""")

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=few_shot_example_prompt,
    suffix="Human: {question}",
    input_variables=["question"],
)

final_prompt = PromptTemplate.from_template("""
    You are a movie geek. Turn the movie title into three emojis that represents the movie.
                                            
    {few_shot_prompt}
    {history}
    Human: {question}
    You:
""")

def load_memory(input):
    memory.load_memory_variables({})["history"]

chain = {
    "few_shot_prompt": few_shot_prompt,
    "history": load_memory,
    "question": RunnablePassthrough(),
} | final_prompt | llm

def invoke_chain(query):
    result = chain.invoke({"question":query})
    memory.save_context({"input":query},{"output":result.content})
    print(result.content)

invoke_chain("Julie and Julia")
invoke_chain("Begin Again")

👩‍🍳📖🍝
🎶🎤❤️


In [21]:
chain.invoke({"question":"What was the first asked movie's title?"})

AIMessage(content='Top Gun')

In [ ]:
import os
from dotenv import load_dotenv
from langchain.globals import set_llm_cache
from langchain.cache import InMemoryCache
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough

load_dotenv()
set_llm_cache(InMemoryCache())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL_NAME"),
    temperature=1.0,
)
memory = ConversationBufferMemory(return_messages=True)

few_shot_example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}"),
])
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=few_shot_example_prompt,
)
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a movie geek. Turn the movie title into three emojis that represents the movie."),
    few_shot_prompt,
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

def load_memory(input):
    return memory.load_memory_variables({})["history"]

chain = RunnablePassthrough.assign(history=load_memory) | final_prompt | llm

def invoke_chain(query):
    result = chain.invoke({"question":query})
    memory.save_context({"input":query}, {"output":result.content})
    print(result.content)

invoke_chain("Rainy day in New York")
invoke_chain("Little Forest")

☔️🌆🎶
🌳🍂🍲


In [12]:
chain.invoke({"question":"What was the first asked movie?"})

AIMessage(content='The first movie you asked about was "Rainy Day in New York."')